# PyVWF on Australia's NEM: seasonal bias correction, end to end

Reproduces the D2 Australia validation: train PyVWF's per-cluster seasonal
wind-speed corrections on 2020–2022 AEMO observations, evaluate on held-out
2023, and apply the seasonal-cycle gate. Headline finding
([docs/findings/pillar_a_au.md](../../docs/findings/pillar_a_au.md)):
**ERA5 over-amplifies South Australia's seasonal cycle, and the correction
compresses it toward observation.**

> **Curve library.** This notebook runs on the **open curve library** for
> reproducibility (redistributable, BSD-3-derived). Its numbers are the
> open-stack ones (fleet cycle-RMSE −8.7%, JJA neutral), which differ from
> the findings doc's primary gate on the **real** licensed library (−10.9%,
> JJA improved). That is expected: see the dual-library table at the end.
> Open-stack curve assignment is p_density-class proxy matching, not per-OEM
> curves.

> **Data wall, stated up front.** Bundled here (~150 KB, derived, attributed:
> Source AEMO; GWPT © Global Energy Monitor CC BY 4.0): monthly SCADA
> aggregates, capacity mask, farm metadata. **Not bundled: you must fetch
> two things before this notebook will run:**
>
> 1. **The ERA5-AU subset.** Requires a (free) Copernicus CDS account of
>    your own: register at cds.climate.copernicus.eu, put your API token in
>    `~/.cdsapirc`, and `pip install cdsapi`. Then, from the repo root:
>    `python scripts/fetch/era5.py --region au_nem` (48 month-requests, ~6–12 GB; the
>    CDS **queue**, not bandwidth, is the real cost: expect hours), followed
>    by `python scripts/era5/combine.py --region au_nem` (reduces to ~29 MB/year,
>    written to `input/era5/AU_daily/` where this notebook looks for it).
> 2. **The open curve library** (`power_curves_open.zip`, distributed
>    separately from this repository: see `examples/data/au_nem/README.md`;
>    it is a redistributable BSD-3-derived set and can also be rebuilt from
>    the NatLabRockies/turbine-models archive using the method in its own
>    README). Unzip it anywhere and set the environment variable
>    `OPEN_CURVES_DIR` to that directory before starting Jupyter (default:
>    `power_curves_open/` at the repo root).
>
> The setup cell below checks both and stops with instructions if either is
> missing.

In [ ]:
# --- Setup: paths + staging. PYVWF_INPUT must be set BEFORE importing vwf. ---
import os, shutil, tempfile
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "configs").is_dir():           # notebook may start in examples/notebooks
    REPO = REPO.parent

ERA5_DAILY  = REPO / "input" / "era5" / "AU_daily"      # from combine_era5_au_daily.py
OPEN_CURVES = Path(os.environ.get("OPEN_CURVES_DIR", REPO / "power_curves_open"))
BUNDLED     = REPO / "examples" / "data" / "au_nem"

missing = [str(p) for p, why in [
    (ERA5_DAILY / "era5_au_daily_2023.nc", "run scripts/fetch/era5.py --region au_nem then scripts/era5/combine.py --region au_nem"),
    (OPEN_CURVES / "power_curves_open_smoothed_cf.csv", "unzip power_curves_open.zip and set OPEN_CURVES_DIR"),
] if not p.is_file()]
assert not missing, f"Missing inputs (see the data-wall cell above): {missing}"

stage = Path(tempfile.mkdtemp(prefix="pyvwf_au_"))
(stage / "observations/turbine" / "AU_NEM").mkdir(parents=True)
(stage / "era5").mkdir()
(stage / "era5" / "AU_daily").symlink_to(ERA5_DAILY)
shutil.copy(OPEN_CURVES / "power_curves_open_smoothed_cf.csv", stage / "power_curves.csv")
shutil.copy(OPEN_CURVES / "models_open.csv", stage / "models.csv")
for f in ["au_nem_scada_monthly_partials.csv", "au_nem_capacity_mask.csv"]:
    shutil.copy(BUNDLED / f, stage / "observations/turbine" / "AU_NEM" / f)
shutil.copy(BUNDLED / "au_nem_md_open.csv",
            stage / "observations/turbine" / "AU_NEM" / "au_nem_md.csv")
os.environ["PYVWF_INPUT"] = str(stage)
print("staging:", stage)

In [ ]:
# --- The region spec: Southern-Hemisphere seasons are EXPLICIT month lists ---
import pandas as pd
from vwf.harness.regions import RegionSpec

SH = {"summer": (12, 1, 2), "autumn": (3, 4, 5), "winter": (6, 7, 8), "spring": (9, 10, 11)}
spec = RegionSpec(
    code="AU-NEM", name="Australia NEM (open stack)", source="aemo-nem",
    obs_level="turbine", obs_unit="farm",
    train_years=(2020, 2022), test_years=(2023,),
    era5_path="era5/AU_daily", bbox=(129.0, 154.0, -44.0, -10.0), file_tag="AU",
    correction_model="affine-wind", cluster_list=(5,),
    time_slices=("fixed", "season"), seasons=SH,
)
md = pd.read_csv(BUNDLED / "au_nem_md_open.csv")
print(f"fleet: {len(md)} farms, {md.capacity.sum()/1e6:.2f} GW; winter = {spec.seasons['winter']} (JJA)")

In [ ]:
# --- Observed seasonal cycles from the bundled AEMO aggregates ---
import matplotlib.pyplot as plt
import numpy as np
from vwf.sources import AEMONemSource

obs = AEMONemSource().load_observations(2023, 2023)          # masks applied inside
obs_long = obs.melt(id_vars=["ID", "year"], var_name="m", value_name="cf")
obs_long["month"] = obs_long["m"].str.replace("obs_", "").astype(int)
obs_monthly = obs_long.pivot_table(index="month", columns="ID", values="cf")

mdx = md.set_index("ID")
def regional_cycle(monthly, ids):
    caps = mdx.loc[ids, "capacity"].astype(float)
    block = monthly[ids]
    w = block.mul(caps, axis=1).sum(axis=1) / block.notna().mul(caps, axis=1).sum(axis=1)
    return w / w.mean()

fig, ax = plt.subplots(figsize=(9, 4.5))
for region in sorted(mdx["region"].unique()):
    ids = [f for f in obs_monthly.columns
           if f in mdx.index and mdx.loc[f, "region"] == region
           and obs_monthly[f].notna().sum() == 12]
    if len(ids) >= 3:
        ax.plot(range(1, 13), regional_cycle(obs_monthly, ids), marker="o",
                label=f"{region} ({len(ids)} farms)")
ax.axvspan(5.5, 8.5, alpha=0.12, color="grey", label="JJA (austral winter)")
ax.set(xlabel="month", ylabel="normalised CF",
       title="Observed 2023 seasonal cycles by NEM region (capacity-weighted)")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()

In [ ]:
# --- Train the corrections (2020-2022) and evaluate on held-out 2023 ---
# Sequential offset fits over daily wind: a couple of minutes.
from vwf.harness.driver import run_evaluate, run_train

out = stage / "runs"
train_dir = run_train(spec, out, run_name="nb")
eval_dir  = run_evaluate(spec, train_dir, out, run_name="nb")
factors = pd.read_csv(train_dir / "factors_season_5.csv")
factors.pivot(index="cluster", columns="season", values="scalar").round(3)

In [ ]:
# --- The gate: corrected cycle must track the OBSERVED cycle better ---
def monthly_from_daily(path):
    cf = pd.read_csv(path); cf["time"] = pd.to_datetime(cf["time"])
    m = cf.set_index("time").resample("ME").mean(); m.index = m.index.month
    return m

unc = monthly_from_daily(eval_dir / "unc_cf.csv")
cor = monthly_from_daily(eval_dir / "cor_cf_season_5.csv")
farms = [f for f in obs_monthly.columns if f in unc.columns
         and obs_monthly[f].notna().sum() == 12]

rows = []
groups = {"FLEET": farms, **{r: [f for f in farms if mdx.loc[f, "region"] == r]
                             for r in sorted(mdx.loc[farms, "region"].unique())}}
for name, ids in groups.items():
    o = regional_cycle(obs_monthly, ids)
    for variant, m in (("uncorrected", unc), ("corrected-season", cor)):
        d = regional_cycle(m, ids).values - o.values
        rows.append({"region": name, "variant": variant, "n": len(ids),
                     "cycle_rmse": float(np.sqrt((d ** 2).mean()))})
gate = pd.DataFrame(rows).pivot_table(index=["region", "n"], columns="variant",
                                      values="cycle_rmse").round(4)
fleet = gate.loc["FLEET"].iloc[0]
verdict = "PASS" if fleet["corrected-season"] < fleet["uncorrected"] else "FAIL"
print(f"GATE (open stack): {fleet['corrected-season']:.4f} vs "
      f"{fleet['uncorrected']:.4f} uncorrected -> {verdict}")
gate

In [ ]:
# --- The finding, visually: South Australia's over-amplified cycle ---
sa = [f for f in farms if mdx.loc[f, "region"] == "SA1"]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(range(1, 13), regional_cycle(obs_monthly, sa), "k-o", lw=2, label="observed")
ax.plot(range(1, 13), regional_cycle(unc, sa), "--s", color="tab:red",
        label="ERA5 uncorrected")
ax.plot(range(1, 13), regional_cycle(cor, sa), "-^", color="tab:blue",
        label="PyVWF corrected (seasonal)")
ax.axvspan(5.5, 8.5, alpha=0.12, color="grey")
ax.set(xlabel="month", ylabel="normalised CF",
       title=f"South Australia ({len(sa)} farms): ERA5 over-amplifies the real cycle")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()

## Dual-library results (from the findings doc)

| Fleet gate (76 farms) | uncorrected | corrected (seasonal) | verdict |
|---|---|---|---|
| **real library** (primary, licensed, `add_models` OEM matching) | 0.0724 | 0.0645 (−10.9%) | PASS |
| **open library** (this notebook; p_density-class proxy) | 0.0672 | 0.0614 (−8.7%) | PASS |

Same verdict, direction, and regional pattern on both stacks: the finding is
robust to curve library *and* matching strategy. The full analysis is in
[docs/findings/pillar_a_au.md](../../docs/findings/pillar_a_au.md): the SA
over-amplification numbers, the selectivity argument (NSW/VIC's mild
degradation is the expected behaviour of an honest seasonal correction), the
n=1-cluster exclusion check, and the curtailment discussion (the claim is
*tracking observable generation*, not pure-resource-bias attribution).